In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class BathtubReconstructor(nn.Module):
    def __init__(self, topo_patch, f, max_iters=20):
        super().__init__()
        self.f = f
        self.max_iters = max_iters
        
        # Ensure topo has 3 dims: [1, Ny_fine, Nx_fine]
        if topo_patch.dim() == 2:
            topo_patch = topo_patch.unsqueeze(0)
        
        # register_buffer ensures this moves to GPU when you call .to('cuda')
        self.register_buffer("topo", topo_patch)

    def forward(self, u_coarse):
        # Dynamically get the device from the input tensor
        device = u_coarse.device 
        bs, n_y, n_x, n_t = u_coarse.shape
        f = self.f
        nf_y, nf_x = n_y * f, n_x * f
        
        # We use .expand(bs, -1, -1) to make sure the single topo buffer 
        # matches the incoming batch size u_coarse without copying memory.
        topo_batch = self.topo.expand(bs, -1, -1)

        # 1. Reshape topo [BS, ny*f, nx*f] -> [BS, ny, f, nx, f]
        topo_folded = topo_batch.view(bs, n_y, f, n_x, f).permute(0, 1, 3, 2, 4)
        z_fine = topo_folded.reshape(bs, n_y, n_x, 1, f*f)
        
        d_target = u_coarse.unsqueeze(-1)

        # Bounds initialization
        h_low = z_fine.min(dim=-1, keepdim=True)[0]
        h_high = z_fine.max(dim=-1, keepdim=True)[0] + d_target + 1e-3

        for _ in range(self.max_iters):
            h_mid = (h_low + h_high) / 2.0
            d_mid = torch.relu(h_mid - z_fine).mean(dim=-1, keepdim=True)
            
            mask = d_mid < d_target
            h_low = torch.where(mask, h_mid, h_low)
            h_high = torch.where(mask, h_high, h_mid)

        h_final = (h_low + h_high) / 2.0
        u_rec = torch.relu(h_final - z_fine) 

        u_rec = u_rec.view(bs, n_y, n_x, n_t, f, f)
        u_rec = u_rec.permute(0, 1, 4, 2, 5, 3).reshape(bs, nf_y, nf_x, n_t)

        return u_rec

In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def test_bathtub_class():
    # --- 1. Configuration ---
    bs = 2
    f = 4
    my, mx = 41, 82  # Coarse Height, Width
    n_t = 88         # Time steps
    
    ny, nx = my * f, mx * f  # Fine Height, Width

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # --- 2. Create Dummy Data ---
    # Coarse depth: [Batch, Height, Width, Time]
    u_coarse = torch.rand(bs, my, mx, n_t).to(device) + 0.1
    
    # Topography: [Batch, Height, Width]
    # In your actual training, this will be your high-res DEM
    topo_fine = torch.randn(bs, ny, nx).to(device)

    print(f"--- Inputs ---")
    print(f"Coarse Depth: {u_coarse.shape}")
    print(f"Fine Topo:    {topo_fine.shape}")

    # --- 3. Initialize the Reconstructor Class ---
    # In your trainer, you do this once at the start
    reconstructor = BathtubReconstructor(topo_fine, f=f, max_iters=20).to(device)

    # --- 4. Run Forward Pass (The Physics Baseline) ---
    u_bathtub = reconstructor(u_coarse)

    print(f"\n--- Output ---")
    print(f"Bathtub HR:   {u_bathtub.shape}")

    # --- 5. Logical Checks ---
    
    # CHECK 1: Shape Match
    assert u_bathtub.shape == (bs, ny, nx, n_t), f"Shape mismatch! Got {u_bathtub.shape}"
    print("✅ Check 1: Output shape matches fine-scale grid.")

    # CHECK 2: Mass Conservation (The Mean Constraint)
    # We pool the fine depth back to coarse to see if it matches the input
    u_bt_permuted = u_bathtub.permute(0, 3, 1, 2) # [BS, T, H_fine, W_fine]
    u_re_coarsened = F.avg_pool2d(u_bt_permuted, kernel_size=f).permute(0, 2, 3, 1) # [BS, H_coarse, W_coarse, T]
    
    diff = torch.abs(u_re_coarsened - u_coarse).mean()
    print(f"✅ Check 2: Mean Difference (Mass Balance): {diff.item():.2e}")
    
    if diff < 1e-4:
        print("\nOVERALL TEST PASSED: The class is ready for training.")
    else:
        print("\nOVERALL TEST FAILED: Mass balance error too high.")

if __name__ == "__main__":
    test_bathtub_class()

--- Inputs ---
Coarse Depth: torch.Size([2, 41, 82, 88])
Fine Topo:    torch.Size([2, 164, 328])

--- Output ---
Bathtub HR:   torch.Size([2, 164, 328, 88])
✅ Check 1: Output shape matches fine-scale grid.
✅ Check 2: Mean Difference (Mass Balance): 6.15e-07

OVERALL TEST PASSED: The class is ready for training.
